# Traffic queues

**Question:** how long is the queue on each arm, and what green time would clear the longest one?

The desk shows North, East, South, and West. **Accept signal plan** stores the suggested seconds. It does not write to a signal controller. That write waits until the department names the controller.

Suggested green seconds are half the queue length in metres, and they stay between 10 and 60. The sample queues are chosen so the plan differs from the greens already running.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityvigilance.runtime import explain, where_this_runs
from cityvigilance.sample import check_event, run_model


In [2]:
print(explain())


Machine: cursor
System: Linux-6.12.94+-x86_64-with-glibc2.39
Python: 3.12.3
NVIDIA tool: not installed
PyTorch installed: False
CUDA available: False
Vision API connected: False
Model source: sample
This notebook runs on: this process, CPU only

The command-centre page is a separate Node process. It serves the desk
and plays the same sample findings in the browser. It does not send
frames to a GPU or to a vision API.

Your own computer's GPU is not used when you watch the demo from Cursor.
If you clone the repo and run it locally, the sample models still use
the CPU only, until you replace one of them with a real model.


In [3]:
event = run_model("traffic", 0)
check_event(event)
print(f"{'arm':<8}{'queue m':>8}{'now s':>8}{'plan s':>8}")
for arm in event["finding"]["arms"]:
    print(f"{arm['name']:<8}{arm['queueMetres']:>8}{arm['currentGreenSeconds']:>8}{arm['greenSeconds']:>8}")


arm      queue m   now s  plan s
North         62      20      31
East          22      20      11
South         34      25      17
West          14      15      10


## Four ticks

The sample stream rotates four queue pictures. The desk plays the same rotation.


In [4]:
for tick in range(4):
    arms = run_model("traffic", tick)["finding"]["arms"]
    north = arms[0]
    print(
        f"tick {tick}: North {north['queueMetres']} m, "
        f"now {north['currentGreenSeconds']}s, plan {north['greenSeconds']}s"
    )


tick 0: North 62 m, now 20s, plan 31s
tick 1: North 58 m, now 20s, plan 29s
tick 2: North 70 m, now 20s, plan 35s
tick 3: North 66 m, now 20s, plan 33s


## Slot for a real model

Return four arms. `greenSeconds` is the plan you want stored. `currentGreenSeconds` is what the controller is running now, when you know it. If you do not know the current plan, still return both fields so the desk can show the difference.


In [5]:
def my_traffic(sequence: int, at: str) -> dict:
    return run_model("traffic", sequence, at)

result = my_traffic(0, "2026-11-02T00:00:00Z")
check_event(result)
arms = result["finding"]["arms"]
assert [arm["name"] for arm in arms] == ["North", "East", "South", "West"]
assert all(10 <= arm["greenSeconds"] <= 60 for arm in arms)
print("traffic event ok", arms[0])


traffic event ok {'name': 'North', 'queueMetres': 62, 'currentGreenSeconds': 20, 'greenSeconds': 31}
